In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

In [ ]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [6]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

answer = rag.rag(query)
print(answer)

answer = rag.rag(query)
print(answer)

answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xbf82a4bbf0984cec84e3fd4d41cd5b7b",
        "span_id": "0x296cd41170a0907b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x267b6cf2303bdd9e",
    "start_time": "2026-07-16T14:35:17.215394Z",
    "end_time": "2026-07-16T14:35:17.219443Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "db7c2f30-4e39-4481-a071-dc2c39e506d8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xbf82a4bbf0984cec84e3fd4d41cd5b7b",
        "span_id": "0xbfffb1bd60dfd98d",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q4

In [3]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True


In [4]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [5]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)


It keeps calling the model inside a `while True` loop and checks whether the model returned any `function_call` items.

- If there is a function call, it runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop and stops.

So the stop condition is: **no function calls in the latest response**.


In [8]:

DB_PATH = "traces.db"

def get_span_names(limit=10):
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT name FROM spans LIMIT ?",
            (limit,)
        ).fetchall()
    return [name for (name,) in rows]

get_span_names()

['search', 'llm', 'rag']

In [9]:
answer = rag.rag(query)
print(answer)


The loop keeps calling the model by checking whether the latest response contains any `function_call` items.

- It sends the current `messages` to the model.
- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls` is still `False`, it breaks out of the loop.

So the model is called again only when it asks for more tool use; the loop stops when it returns a final message with no function calls.


## Q5

In [11]:
sqltqry = """ 
    SELECT name, end_time - start_time as duration 
    FROM spans
    where name <> 'rag'
    """

with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(sqltqry).fetchall()

[(name, duration) for (name, duration) in rows]

[('search', 3311732),
 ('llm', 3341848977),
 ('search', 7856093),
 ('llm', 2945756325)]

## Q6

In [12]:
for i in range(2):
    answer = rag.rag(query)
print(answer)


sqltqry = """ 
    SELECT name, input_tokens 
    FROM spans
    where name = 'llm'
    """

with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(sqltqry).fetchall()

[(name, duration) for (name, duration) in rows]

It keeps a `while True` loop going, and after each model call it checks whether the response contains any `function_call` items.

- If there are function calls, the code runs the tools, appends the tool outputs to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the model’s response**.


[('llm', 7111), ('llm', 7111), ('llm', 7111), ('llm', 7111)]